CASO PRATICO: USARE GPT VIA API PER GENERARE RIASSUNTI

L'API GPT può essere usata non solo per chiacchierare, ma anche per costruire un sistema professionale di riassunto testi.

Immagina di dover leggere migliaia di mial al giorno, GPT può farlo per noi, ma solo se sappiamo come impartire l'ordine corretto e soprattutto, come gestire i momenti in cui la connessione o il server, decidono di non collaborare.

- System Prompt: configurazione del comportamento della persona, l'identità del modello è tutto.
- Gestione Errori: strategie di retry e backoff esponenziale. Strategia matematiche per gestire gli errori di rete senza far craschare la nostra app.
- Formattazione output: controllo estetico, forzando il modello a parlare con markdown, rendendo i riassunti pronti per essere pubblicati 

Prendiamo un testo da un programma (es. in Python), inviamo il testo a un modello GPT tramite l'API OpenAI e gli chiediamo di restituire una versione più corta che conservi le informazioni importanti.
testo originale -> programma python -> openAI API -> modello GPT -> riassunto -> programmma python

E' simila a quello fatto con Hugging-Face, cambia il provider
OpenAI oggi indica la Responses API come interfaccia principale per richieste dirette di generazione testo.

Esempio

from openai import OpenAI
client = OpenAI()
testo = """
L'intelligenza artificiale sta modificando profondamente i processi aziendali. Le aziende stanno introducendo sistemi di automazione per analizzare documenti, classificare email, prevedere la domanda e supportare le decisioni operative.
"""
response = client.responses.create(
    model="gpt-6-astra",
    input=f"""
    Riassumi il seguente testo in massimo 3 frasi.

    TESTO:
    {testo}
    """
)
print(response.output_text)

l'istruzione client.responses.create invia la richiesta al modello, la risposta testuale si recupera con response.output_text

Il VERO lavoro lo fa il prompt
"riassumi il testo" o "riassumi il testo in massimo 5 punti. Mantieni dati numerici, date, nomi, decisioni prese, problemi aperti. Elimina ripetizioni, formule di cortesia e dettagli non operativi"
Quindi lo stesso testo può produrre riassunti molto diversi in funzione dell'istruzione.

RIASSUNTO
riduco il testo mantenendo il significato
ESTRAZIONE
recupero campi specifici
CLASSIFICAZIONE
assegno una categoria

RIASSUNTO
(riduco il testo mantenendo il significato)
Esistono due modi per fare riassunto:
    1) riassunto estrattivo: vengono selezionate parti già presenti nel testo
    2) riassunto astrattivo: tipico degli LLM, il modello (in questo caso GPT) non deve necessariamente copiare frasi, può generare una nuova formulazione. Tutto questo è portente ma introduce il rischio che il modello possa reinterpretare o introdurre informazioni non presenti. 
    Per documenti importanti aggiungere istruzioni del tipo: Non aggiungere informazioni non presenti. Mantieni invariati numeri, date e codici. Se u n'informazione manca, non dedurla.

E se il testo da riassumero è lungo?
Ogni modello ha una context window, cioè un limite alla quantità di token che può ricevere ed elaborare. La Responses API può anche gestire strategie di truncation, ma normalmente non vuoi affidarti a una semplice eliminazione automatica di parti di testo, perchè potresti perdere proprio le informazioni importanti.
Per un documento lungo è preferibile fare map-reduce:
DOCUMENTO -> divisione in blocchi -> riassunto blocco 1 - riassunto blocco 2 - riassunto blocco n -> riassunto dei riassunti.

Puoi anche inviare direttamente dei file
La Responses API support input testuali ma anche file in alcune modalità. La documentazione OpenAI mostra, per esempio, l'invio di un PDF insieme all'istruzione "analizza e riassumi in punti principali"

Parametro Temperature
Per un riassunto aziendale, probabilmente, non vuoi molta creatività. 
Una temperatura bassa tende a produrre risposte pià focalizzate e meno variabili; valori più alti rendono la generazione più casuale. OpenAI documenta 

Identità digitale
Il Ruolo del System Prompt

Definire l'identità del modello
Nelle API di messaggistica moderna, il 'System Prompt' è lo strumento che stabilisce le regole di base e il tono della conversazione prima ancora che l'utente invii il primo input (apra bocca). Non è una semplice istruzione, e la definizione delle regole del contesto
Per un compito di riassunto, non basta chiedere di 'riassumere'. Dobbiamo definire chi sta scrivendo e per quale pubblico, trasformando il modello in un esperto editore.
Il vocabolario che la rete utilizza cambia completamente in base al contesto in cui deve operare.
Senza il Ruolo, sarebbe come chiedere ad un attore di recitare senza dirgli che personaggio interpreta.
Il system prompt è il copione che diamo a GPT prima che l'utente apra bocca.
E' uno strumento che trasforma un completatore di frasi in uno specialista della sintesi.

Ma come fa il copione a influenzare la matematica del modello?
Steering e Condizionamento
Guidare la distribuzione probabilistica
- Persona: assegnare un ruole specifico (es. 'sei un ricercatore scientifico esperto di sintesi') per attivare vocabolari tecnici pertinenti.
- Vincoli di Stile: specificare il tono (formale, conciso, persuasivo) per allineare l'output agli obbiettivi aziendali
- Limitazione del Dominio: istruire il modello a ignorare informazioni esterne al testo fornito per ridurre le allucinazioni.
Il system prompt agisce come un vincolo di condizionamento, che restinge lo spazio delle risposte possibili verso la distribuzione desiderata. Riduce la distribuzione di probabilità. 
E' come dare al modello una bussola semantica, portandolo in una zona dello spazio latente dove le allucinazioni sono meno probabili e dove il tono è quello desiderato

Tecniche di Role-Playing
Non limitarti a dire: "riassumi questo...". Inserire nel system prompt informazioni sul target di lettura (es. 'scrivi per un pubblico di non esperti'). Definisci il ruolo ('sei una analista finanziario')
Possiamo anche inserire dei vincoli negativi, cosa 'non fare' come 'non superare i 3 paragrafi' è fondamentale per il controllo di qualità (negative constraints).
Il bilanciamento tra un system prompt forte e una temperatura bassa (es. 0.2) garantisce riassunti fattuali e ripetibili.

Quanto può essere profondo il condizionamento comportamentale.
Igegneria del Comportamento
Dall'istruzione al risultato
L'obbiettivo finale è minimizzare la divergenza tra ciò che il modello farebbe normalmente e ciò che serve a noi.
Il condizionamento iniziale tramite messaggi di sitema, permette di risparmiare token nei messaggi utente successivi.
Più l'identità è granulare, più il modello sarà in grado di gestire sfumature semantiche complesse senza deviare dal task primario.
Questo approccio ci permette di risparmiare token, invece di ripetere le istruzioni in ogni messaggio dell'utente le cristalliziamo, una volte per tutte, nel messaggio di sistema.

Ma cosa succede quando, nonostante il prompt perfetto, l'API smette di rispondere?
(ben venuti nel mondo reale del cloud computer)
Lavorare con API come quella di OpenAI o Antropich, significa accettare l'imprevisto, server sovraccarico, un picco di traffico mondiale, o un micro blocco nella nostra connessione.
Dobbiamo distinguere tra un problema fatale, come una chiamata API errata, e un problema transitorio, risolvibele semplicemente avendo un po' di pazienza. 
Un'applicazione professionale deve implementare logiche di gestione degli errori che distringuano tra fallimenti fatali e problemi transitori.

Il problema transitorio più comune ha un nome preciso: errore 429

- Error 429: segnale che abbiamo superato il numero di richieste o di token consentiti per minuto dal provider.
- Retry Logic: non interrompere il processo al primo errore, ma tentare nuovamente la chiamata dopo un intervallo di tempo
- Exponential Backoff: aumentare progressivamente il tempo di attesa tra un tentativo e l'altro per non sovraccaricare ulteriomente il server e che peggiorerebbe la situazione.
Quindi non riprovare subito, aspettiamo un secondo, poi 2, poi 8 e così via.
- L'aggiunta di un elemento casuale (jitter) evita che migliaia di client tentino il retry nello stesso identico istante, creando un nuovo crash.

Dobbiamo però mettere dei limiti alla pazienza.
Non vogliamo che il nostro programma aspetti all'infinito, impostiamo un numero massimo di tentativi (es. 3-5) per evitare loop infiniti che consumerebbero risorse inutilmente (max retries).
Inoltre, configurare tempi di attesa massimi per la risposta dell'API per evitare che il thread Python rimanga appeso indefinitamente (timeout handling).
Tracciare gli errori API è fondamentale per monitorare la salute dell'infrastruttura e i costi operativi nel tempo (logging).
Se l'AI fallisce nel generare un riassunto, dobbiamo sapere esattamente perchè è successo, questo ci permette di monitorare non solo la qualità tecnica ma anche l'impatto economico.

Vediamo come la matematica ci aiuta a gestire questa attesa intelligente.
Algoritmi di Attesa
La matematica della pazienza
Senza backoff, una pipeline di riassunti massivi fallirebbe non appena si incontra un picco di traffico.
L'uso di librerie nate apposta per questo, come 'tenacity', permette di decorare le funzioni Python rendendole intrinsecamente resilienti agli errori di rete.
La logica è semplice, se la funzione va in errore, la stessa va a dormire per un tempo che cresce esponenzialmente.
Questa è la differenza tra un software che si rompe al primo soffio di vento, ed uno che sa piegarsi per poi rialzarsi.

Ora che la pipeline è sicura, occupiamoci di come deve apparire il risultato finale

Controllo del Formato
Dalla stringa all'output strutturato.
Un riassunto è utile solo se è leggibile a colpo d'occhio e integrabile.
Se GPT ci restituisce un muro di testo, abbiamo fallito metà del compito
Dobbiamo forzare il modello a utilizzare uno standard visivo e lo stanrdard del web oggi è il Marckdown, utile per facilitare la visualizzazione.
Nel prompt dobbiamo combinare istruzioni di formattazione precise per ottenere elenchi puntati, tabelle o sezioni markdown specifiche.
Ogni informazione deve avere il suo posto prestabilito

Markdown e Indicatori
Strutturare l'informazione visiva
- Bullet Points: chiedere esplicitamente un elenco puntato per evidenziare i concetti chiave senza dispersioni narrative. Sono le ancore per l'occhio del lettore.
- Sezioni Titolate: usare header Markdown (##) per dividere il riassunto in sezioni logiche: introduzione, corpo (analisi) e conclusioni
- Lenght Control: impostare vincoli sulla lunghezza (es. massimo 100 parole) invece di sperare nel comportamento di default. Senza questo limite il modello continuerà a essere prolisso, perchè statisticamente è più facile continuare a scrivere che fermarsi al momento giusto.
- La precisione nel formato riduce la necessità di post-processing manuale sui test generati.

Per chi cotruisce software puro, a volte il testo non basta. 
Serve un formato JSON, possiamo istruire GPT a scrivere il riassunto dentro una chiave specifica del dizionario JSON. In questo modo il software non dovrà fare parsing del testo, ma potrà leggere direttamente il dato strutturato.

Un'altro formato utile è l'highlighting
Utilizzare il grassetto per enfatizzare le entità nominate (nomi propri) o le date importanti all'interno del riassunto.
Trasforma un testo piatto in una mappa informativa dove i dati critici saltano subito agli occhi.

Ma come succese, logicamente, quando sintetizziamo un testo?

La Logica della Sintesi.
Sintetizzare significa massimizzare l'entropia informativa.
Dobbiamo condensare la massima quantità di significato nel minor numero di token possibili.
Un buon riassunto deve catturare la massimo informazione semantica con il minor numero di token possibili.
Il controllo del formato, non è quindi solo estetica, è il vincolo che obbliga il modello a fare selezione, scartando il rumore e focalizzarsi sul segnale principale. scartando gli aggettivi innutili e tenendo solo i verbi di azione e i sotantivi chiave.
Un riassunto ben formattato è la prova che il modello ha davvero distillato il contenuto originale.

LM Studio permette di creare un server in locale con un modello scelto.

In [2]:
import os
from dotenv import load_dotenv
from openai import OpenAI, RateLimitError
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type

# PUNTO DI INGRESSO DELLA CONFIGURAZIONE
# Carichiamo le variabili di ambiente definite nel file esterno '.env'.
# Questa pratica è fondamentale per la sicurezza e la flessibilità: 
# permette di modificare parametri come l'indirizzo del server o il modello 
# senza dover alterare il codice logico del programma.
load_dotenv()

# ACQUISIZIONE DEI PARAMETRI DI CONNESSIONE
# Utilizziamo la libreria 'os' per leggere i valori caricati da '.env'.
# - base_url: L'indirizzo di rete dove LM Studio espone le proprie API (es. localhost:1234).
# - model_name: Il nome identificativo del modello che abbiamo caricato e attivato in LM Studio.
# - api_key: Sebbene LM Studio spesso non richieda autenticazione, la libreria OpenAI 
#   necessita di una stringa (anche casuale) per inizializzare il client correttamente.
base_url = os.getenv("OPENAI_BASE_URL")
model_name = os.getenv("OPENAI_MODEL_NAME")
api_key = os.getenv("OPENAI_API_KEY")

# INIZIALIZZAZIONE DEL CLIENT DI COMUNICAZIONE
# Qui creiamo un'istanza della classe 'OpenAI'. Il parametro fondamentale è 'base_url',
# che istruisce il client a non contattare i server ufficiali di OpenAI, ma a reindirizzare 
# ogni richiesta verso il nostro server locale gestito da LM Studio.
client = OpenAI(
    base_url=base_url,
    api_key=api_key
)

# LOGICA DI RETRY ESPONENZIALE CON TENACITY
# Utilizziamo un decoratore per gestire automaticamente i tentativi di riconnessione.
# - retry_if_exception_type: Interveniamo solo se l'errore è un RateLimitError (429).
# - wait_exponential: Applichiamo un'attesa che cresce esponenzialmente (es. 1, 2, 4 secondi).
# - stop_after_attempt: Limitiamo il numero massimo di tentativi per evitare cicli infiniti.
@retry(
    retry=retry_if_exception_type(RateLimitError),
    wait=wait_exponential(multiplier=1, min=2, max=10),
    stop=stop_after_attempt(5),
    before_sleep=lambda retry_state: print(f"Server occupato. Nuovo tentativo tra pochi secondi... (Tentativo n. {retry_state.attempt_number})")
)
def chiamata_api_riassunto(testo):
    """
    Esegue la chiamata effettiva all'API. Questa funzione è isolata per permettere
    al decoratore 'tenacity' di monitorare specificamente questa operazione.
    """
    return client.chat.completions.create(
        model=model_name,
        messages=[
            {
                "role": "system", 
                "content": "Sei un assistente specializzato nel riassumere testi complessi in punti chiave chiari e precisi."
            },
            {
                "role": "user", 
                "content": f"Analizza e riassumi il seguente testo:\n\n{testo}"
            }
        ],
        temperature=0.3,
        max_tokens=500
    )

def esegui_riassunto(testo):
    """
    Funzione principale che coordina la richiesta di riassunto.
    Gestisce il risultato della chiamata API e le eventuali eccezioni finali.
    """
    
    # Informiamo l'utente sull'avvio del processo.
    print(f"Elaborazione avviata con il modello '{model_name}'...")
    
    try:
        # Invochiamo la funzione protetta dal sistema di retry.
        risposta = chiamata_api_riassunto(testo)
        
        # NAVIGAZIONE NELL'OGGETTO DI RISPOSTA
        # Estraiamo il contenuto testuale dalla struttura restituita dal modello.
        return risposta.choices[0].message.content
        
    except RateLimitError:
        return "Errore: Il server è rimasto sovraccarico dopo tutti i tentativi di recupero."
    except Exception as e:
        # GESTIONE DELLE ECCEZIONI GENERICHE
        # Catturiamo errori critici per evitare il crash del programma.
        return f"Si è verificato un errore imprevisto: {e}"

# BLOCCO DI ESECUZIONE PRINCIPALE
if __name__ == "__main__":
    # Testo sorgente per il test della funzione.
    testo_per_analisi = (
        "L'intelligenza artificiale (IA) è il ramo dell'informatica che si occupa dello sviluppo di "
        "macchine in grado di eseguire compiti che normalmente richiedono l'intelligenza umana. "
        "Questi compiti includono l'apprendimento, il ragionamento, la risoluzione di problemi, la "
        "percezione e la comprensione del linguaggio naturale. Negli ultimi anni, grazie alla "
        "disponibilità di grandi quantità di dati e alla potenza di calcolo crescente, l'IA ha "
        "compiuto passi da gigante, diventando parte integrante di molte tecnologie quotidiane."
    )
    
    # Esecuzione del processo di riassunto.
    risultato_sintesi = esegui_riassunto(testo_per_analisi)
    
    # PRESENTAZIONE DEI RISULTATI A VIDEO
    # Utilizziamo una formattazione semplice e pulita per il terminale.
    print("\n" + "-"*50)
    print("ESITO DELL'ELABORAZIONE (RIASSUNTO):")
    print("-"*50)
    print(risultato_sintesi)
    print("-"*50)

Elaborazione avviata con il modello 'openai/gpt-oss-20b'...

--------------------------------------------------
ESITO DELL'ELABORAZIONE (RIASSUNTO):
--------------------------------------------------
**Punti chiave**

- **Definizione di IA:** ramo dell’informatica che crea macchine capaci di svolgere compiti normalmente riservati all’intelligenza umana (apprendimento, ragionamento, problem‑solving, percezione e comprensione del linguaggio naturale).  
- **Evoluzione recente:** l’IA ha fatto progressi significativi grazie a:  
  - disponibilità massiccia di dati;  
  - aumento della potenza di calcolo.  
- **Impatto pratico:** oggi le tecnologie IA sono integrate in molte applicazioni quotidiane, rendendole parte integrante del nostro ambiente digitale.
--------------------------------------------------
